# IMPACT Entity Data Module — Demo Walkthrough

This notebook walks through the full IMPACT Entity Data Module pipeline:

1. **Data Generation** — Create sample lending data (SQLite, Parquet, CSV)
2. **Facility Pipeline** — Single-entity pipeline with joins, transforms, validations, and sub-entities
3. **Obligor Pipeline** — 3-level nesting: Obligor → Facility → Collateral
4. **Runtime Parameters** — Override config defaults at execution time
5. **Validation Reports** — Diagnostic output for data quality checks
6. **Config Merge** — Combine standardized configs with custom overrides

The pipeline follows 5 stages: **Load → Join → Transform (+ Filter) → Validate → Build**.

In [ ]:
import os
os.chdir(os.path.join(os.path.dirname(os.getcwd()), ""))  # ensure we're at project root
# If running from notebooks/, go up one level
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(f"Working directory: {os.getcwd()}")

## 1. Generate Sample Data

The demo uses simulated lending data saved in three formats:
- **SQLite** — `lending.db` with `obligor_main` and `facility_main` tables
- **Parquet** — `collateral.parquet`
- **CSV** — `rating_overrides.csv`

In [ ]:
import sys
sys.path.insert(0, "scripts")
from generate_sample_data import main as generate_data
generate_data()

In [ ]:
# Peek at the generated data
import pandas as pd
import sqlite3

conn = sqlite3.connect("data/sample/lending.db")
print("=== Facility Main (first 5 rows) ===")
display(pd.read_sql("SELECT * FROM facility_main LIMIT 5", conn))

print("\n=== Obligor Main (first 5 rows) ===")
display(pd.read_sql("SELECT * FROM obligor_main LIMIT 5", conn))
conn.close()

print("\n=== Collateral (Parquet, first 5 rows) ===")
display(pd.read_parquet("data/sample/collateral.parquet").head())

print("\n=== Rating Overrides (CSV, first 5 rows) ===")
display(pd.read_csv("data/sample/rating_overrides.csv").head())

## 2. Facility Pipeline — Single Entity

The Facility config (`configs/demo/facility_demo.yaml`) demonstrates:
- **3 source types** in one pipeline: SQLite (primary), Parquet, CSV
- **Joins**: 1-to-many (collateral → nested DataFrame) and 1-to-1 (rating overrides)
- **Pre-filters**: applied before field processing (raw column names)
- **Post-filters**: applied after field processing (processed field names, `@param` references)
- **Source fields** (Pass 1): rename, expression, cast, fill_na
- **Derived fields** (Pass 2): pandas eval and row-wise lambdas
- **Inline validations**: not_null, unique, range, expression
- **Sub-entities**: `collateral_items` with `entity_ref: Collateral`
- **Temp fields**: intermediate calculations excluded from the entity class

In [ ]:
from impact.entity.pipeline import EntityPipeline

result = EntityPipeline("configs/demo/facility_demo.yaml").run()

print(f"Entity class:    {result.entity_class.__name__}")
print(f"Entities:        {len(result.entities)}")
print(f"DataFrame shape: {result.dataframe.shape}")
print(f"Metadata:        {result.metadata}")
print(f"Sub-entity classes: {result.sub_entity_classes}")

In [ ]:
# Inspect the dynamically-created entity class
Facility = result.entity_class
print(f"Class name:   {Facility.__entity_name__}")
print(f"Primary key:  {Facility.__primary_key__}")
print(f"Fields:       {[f.name for f in Facility.__entity_fields__]}")

# Look at one entity instance
facility = result.entities[0]
print(f"\n--- First Facility ---")
print(f"  facility_id:        {facility.facility_id}")
print(f"  obligor_id:         {facility.obligor_id}")
print(f"  product_category:   {facility.product_category}")
print(f"  commitment_amount:  {facility.commitment_amount:,.2f}")
print(f"  outstanding_balance:{facility.outstanding_balance:,.2f}")
print(f"  utilization_rate:   {facility.utilization_rate:.4f}")
print(f"  days_to_maturity:   {facility.days_to_maturity}")
print(f"  rating_override:    {facility.rating_override}")
print(f"  collateral_items:   list[Collateral] with {len(facility.collateral_items)} items")

In [ ]:
# View the processed DataFrame (temp fields like 'available_capacity' are already dropped)
display(result.dataframe.head(10))

### Sub-Entities: Nested Collateral Items

During the join stage, one-to-many joins produce a nested `pd.DataFrame` in each row's `collateral_items` cell. The sub-entity processor then validates, transforms, and converts each nested DataFrame into a `list[Collateral]` — so by the time you access `facility.collateral_items`, it's a list of `Collateral` dataclass instances, not a DataFrame.

The sub-entity config (`collateral_demo.yaml`) defines the fields, validations, and derived fields for each collateral record.

In [ ]:
# Find a facility with collateral items
for entity in result.entities:
    items = entity.collateral_items
    if isinstance(items, list) and len(items) > 0:
        print(f"Facility {entity.facility_id} has {len(items)} collateral items (Collateral instances):")
        for item in items:
            print(f"  - {item.collateral_type}: ${item.collateral_value:,.2f} ({item.value_bucket})")
        print(f"  Total collateral value: ${entity.total_collateral_value:,.2f}")
        break

## 3. Obligor Pipeline — 3-Level Nesting

The Obligor config (`configs/demo/obligor_demo.yaml`) demonstrates:
- **Pre-joins**: enrich facility rows with collateral and ratings *before* nesting under obligor
- **3-level hierarchy**: Obligor → `facilities: list[Facility]` → `collateral_items: list[Collateral]`
- **Aggregation lambdas**: `total_commitment`, `weighted_avg_rate`, `total_collateral_value`
- **Cross-field derived**: `debt_to_asset_ratio`, `leverage_flag`

In [ ]:
obligor_result = EntityPipeline("configs/demo/obligor_demo.yaml").run()

print(f"Entity class:    {obligor_result.entity_class.__name__}")
print(f"Obligors:        {len(obligor_result.entities)}")
print(f"Sub-entity classes: {obligor_result.sub_entity_classes}")
print(f"Metadata:        {obligor_result.metadata}")

In [ ]:
# Navigate the 3-level hierarchy: Obligor → Facility → Collateral
obligor = obligor_result.entities[0]

print(f"=== {obligor.legal_name} ({obligor.obligor_id}) ===")
print(f"  Industry:       {obligor.industry}")
print(f"  Country:        {obligor.country}")
print(f"  Revenue:        ${obligor.revenue:,.2f}")
print(f"  Leverage flag:  {obligor.leverage_flag}")
print(f"  Debt/Asset:     {obligor.debt_to_asset_ratio:.4f}")
print(f"  # Facilities:   {obligor.facility_count}")
print(f"  Total commitment:    ${obligor.total_commitment:,.2f}")
print(f"  Total outstanding:   ${obligor.total_outstanding:,.2f}")
print(f"  Weighted avg rate:   {obligor.weighted_avg_rate:.4f}")
print(f"  Total collateral:    ${obligor.total_collateral_value:,.2f}")

# Drill into facilities (list[Facility] after sub-entity processing)
facilities = obligor.facilities
print(f"\n  --- Facilities ({len(facilities)} sub-entity instances) ---")
for fac in facilities[:3]:
    print(f"    {fac.facility_id}: commitment=${fac.commitment_amount:,.2f}, balance=${fac.outstanding_balance:,.2f}")

In [ ]:
# Summary view of all obligors
summary = obligor_result.dataframe[[
    "obligor_id", "legal_name", "industry", "facility_count",
    "total_commitment", "total_outstanding", "debt_to_asset_ratio", "leverage_flag"
]]
display(summary)

## 4. Runtime Parameters

Parameters defined in the YAML config are defaults. Override them at runtime via `pipeline.run(parameters={...})`.

The facility demo config has:
- `snapshot_date: "2025-12-31"` — used in SQL WHERE clauses via `{param}` syntax
- `active_product: "TERM_LOAN"` — used in post-filter via `@param` syntax
- `source_table: "facility_main"` — parameterized table name in SQL

In [ ]:
# Default: active_product = "TERM_LOAN" (from config)
result_term = EntityPipeline("configs/demo/facility_demo.yaml").run()
print(f"Default (TERM_LOAN): {len(result_term.entities)} facilities")
print(f"  Products: {result_term.dataframe['product_category'].unique().tolist()}")

# Override: filter to REVOLVER instead
result_rev = EntityPipeline("configs/demo/facility_demo.yaml").run(
    parameters={"active_product": "REVOLVER"}
)
print(f"\nOverride (REVOLVER): {len(result_rev.entities)} facilities")
print(f"  Products: {result_rev.dataframe['product_category'].unique().tolist()}")

## 5. Validation Report

Every pipeline run produces a `ValidationReport` with detailed diagnostics. Validations use two severity levels:
- **`error`** — halts the pipeline immediately (e.g., null primary keys)
- **`warning`** — logs and continues (e.g., out-of-range values)

In [ ]:
report = result.validation_report

print(f"Has errors:   {report.has_errors}")
print(f"Has warnings: {report.has_warnings}")
print(f"Error count:  {report.error_count}")
print(f"Warning count:{report.warning_count}")

print(f"\n--- All validation results ---")
for r in report.results:
    status = "PASS" if r.passed else r.severity.upper()
    print(f"  [{status:>7}] {r.rule_type:12} | {r.field_name or 'global':20} | {r.message}")

In [ ]:
# What happens when a validation with severity=error fails?
from impact.common.exceptions import ValidationError

try:
    # Add a global validation that will fail
    from impact.entity.config.schema import EntityConfig
    import yaml

    with open("configs/demo/facility_demo.yaml") as f:
        config_dict = yaml.safe_load(f)

    # Add a validation that no facility can pass
    config_dict["validations"] = [{
        "type": "expression",
        "rule": "commitment_amount > 999_999_999",
        "message": "Commitment must exceed $1B (intentionally impossible)",
        "severity": "error",
    }]

    config = EntityConfig.model_validate(config_dict)
    EntityPipeline(config=config).run()

except ValidationError as exc:
    print("Pipeline halted with ValidationError!")
    print(f"  Error count: {exc.report.error_count}")
    print(f"\n--- Diagnostic detail ---")
    print(exc.report.format_detail()[:500])  # first 500 chars

## 6. Config Merge — Custom Spaces

IMPACT provides standardized primary configs. Users create **sparse custom configs** that add fields in named **"spaces"** — isolated from primary fields. Custom fields are stored in a `spaces` dict on the entity (plain dicts, not dataclasses), keeping them clearly distinct from sub-entities.

```
┌───────────────────────────────────────────────────────────┐
│ Facility (dataclass)                                      │
│                                                           │
│   .facility_id = "FAC-001"   ─┐                           │
│   .commitment_amount = 1M     │ direct attributes         │
│   .utilization_rate = 0.75   ─┘ (primary config fields)   │
│                                                           │
│   .spaces = {                                             │
│     "risk": {                    ← plain dict, NOT a      │
│         "risk_score": 0.85,       dataclass — clearly     │
│         "risk_grade": "HIGH",     distinct from sub-      │
│     },                            entities like Collateral │
│     "reporting": {                                        │
│         "report_flag": True,                              │
│         "report_category": "LARGE",                       │
│     },                                                    │
│   }                                                       │
│                                                           │
│   .collateral_items = [          ← sub-entities are       │
│       Collateral(...),             separate entity         │
│       Collateral(...),             instances with their    │
│   ]                                own primary keys        │
└───────────────────────────────────────────────────────────┘
```

**Spaces** = flat dicts of extra fields on the same entity.
**Sub-entities** = lists of separate entity instances.

In [ ]:
from impact.entity.pipeline import EntityPipeline
import dataclasses

# --- Single custom space via the unified pipeline interface ---
result = EntityPipeline(
    config="configs/demo/facility_demo.yaml",
    custom={"risk": "configs/demo/custom_risk.yaml"},
).run()
entity = result.entities[0]

print("=== Single Custom Space (risk) ===")
print(f"Entity class:      {result.entity_class.__name__}")
print(f"Is dataclass:      {dataclasses.is_dataclass(entity)}")
print(f"Entities:          {len(result.entities)}")
print(f"\nPrimary fields (direct attributes):")
print(f"  facility_id:       {entity.facility_id}")
print(f"  commitment_amount: {entity.commitment_amount:,.2f}")
print(f"\nSpace fields (transparent access for unique names):")
print(f"  entity.risk_score: {entity.risk_score:.4f}")
print(f"  entity.risk_grade: {entity.risk_grade}")
print(f"\nSpaces dict (explicit access):")
print(f"  entity.spaces = {entity.spaces}")

### Multiple Custom Spaces

Each custom config occupies its own space. Same field names across different spaces are allowed — they live in separate layers.

In [ ]:
# --- Multiple custom spaces ---
result = EntityPipeline(
    config="configs/demo/facility_demo.yaml",
    custom={
        "risk": "configs/demo/custom_risk.yaml",
        "reporting": "configs/demo/custom_reporting.yaml",
    },
).run()
entity = result.entities[0]

print("=== Multiple Custom Spaces (risk + reporting) ===")
print(f"Entities: {len(result.entities)}")
print(f"\nPrimary field:     entity.facility_id = {entity.facility_id}")
print(f"Risk field:        entity.risk_score = {entity.risk_score:.4f}")
print(f"Reporting field:   entity.report_flag = {entity.report_flag}")
print(f"\nAll spaces:")
for space_name, space_dict in entity.spaces.items():
    print(f"  {space_name}: {space_dict}")

# Qualified access for disambiguation
print(f"\nQualified access:  entity['risk.risk_score'] = {entity['risk.risk_score']:.4f}")
print(f"                   entity['reporting.report_flag'] = {entity['reporting.report_flag']}")

### Space Management — `primary_only()`, `drop_space()`, `select_space()`

All methods return **new copies** — the original entity is never mutated.

In [ ]:
# primary_only() — strip all custom spaces
primary = entity.primary_only()
print("=== primary_only() ===")
print(f"  facility_id:  {primary.facility_id}")
print(f"  commitment:   {primary.commitment_amount:,.2f}")
print(f"  has spaces:   {hasattr(primary, 'spaces')}")
print(f"  is dataclass: {dataclasses.is_dataclass(primary)}")

# drop_space() — remove one space, keep the rest
reduced = entity.drop_space("reporting")
print(f"\n=== drop_space('reporting') ===")
print(f"  risk_score:   {reduced.risk_score:.4f} (risk space still present)")
print(f"  spaces:       {list(reduced.spaces.keys())}")

# select_space() — keep only one space
risk_only = entity.select_space("risk")
print(f"\n=== select_space('risk') ===")
print(f"  risk_score:   {risk_only.risk_score:.4f}")
print(f"  spaces:       {list(risk_only.spaces.keys())}")

# Original is unchanged
print(f"\n=== Original entity unchanged ===")
print(f"  entity.risk_score:  {entity.risk_score:.4f}")
print(f"  entity.report_flag: {entity.report_flag}")
print(f"  entity.spaces keys: {list(entity.spaces.keys())}")

# to_dict() — flat dict of all fields
d = entity.to_dict()
print(f"\n=== to_dict() ===")
print(f"  Keys: {list(d.keys())}")

### Auto-naming from Filenames

When no explicit space name is provided, the space name is derived from the config filename stem. You can also pass a list of paths.

In [ ]:
# Auto-naming: single path → space name from filename stem
result_auto = EntityPipeline(
    config="configs/demo/facility_demo.yaml",
    custom="configs/demo/custom_risk.yaml",       # space = "custom_risk" (from filename)
).run()
space_fields = [f for f in result_auto.entity_class.__entity_fields__ if hasattr(f, 'metadata')]
print("Auto-named space:", list(result_auto.entities[0].spaces.keys()))

# Auto-naming: list of paths → each named from its filename
result_list = EntityPipeline(
    config="configs/demo/facility_demo.yaml",
    custom=[
        "configs/demo/custom_risk.yaml",           # space = "custom_risk"
        "configs/demo/custom_reporting.yaml",      # space = "custom_reporting"
    ],
).run()
print("List auto-named spaces:", list(result_list.entities[0].spaces.keys()))

# Advanced: merge_configs() is still available for programmatic use
from impact.entity.config.merger import merge_configs
config = merge_configs(
    primary="configs/demo/facility_demo.yaml",
    custom={"risk": "configs/demo/custom_risk.yaml"},
)
result_manual = EntityPipeline(config=config).run()  # config carries config_path automatically
print("Direct merge_configs:", list(result_manual.entities[0].spaces.keys()))

## Summary

The Entity Data Module pipeline:

| Stage | What happens |
|---|---|
| **Load** | Sources loaded from SQLite, Parquet, CSV (or Snowflake). Parameters interpolated via `{param}`. |
| **Join** | Sources combined: 1-to-1 (flat merge) or 1-to-many (nested DataFrames). Pre-joins supported. |
| **Pre-filter** | Row-level filters on raw columns before field processing. |
| **Transform** | Pass 1: source fields (rename/expr → cast → fill_na). Pass 2: derived fields (eval/lambda). Use `@param` for external parameters in both eval and lambda expressions. |
| **Post-filter** | Row-level filters on processed fields. `@param` syntax for runtime values. |
| **Validate** | Inline field validations + global rules. Error = halt, Warning = continue. |
| **Sub-entity** | Nested DataFrames processed through sub-entity configs (recursive). |
| **Build** | Dynamic dataclass created from fields. Temp fields dropped. Entities instantiated. |

**Unified pipeline interface:**
```python
EntityPipeline(
    config="primary.yaml",                           # required
    custom={"risk": "risk.yaml"},                    # optional: parent custom spaces
    sub_entity_custom={"Collateral": "coll.yaml"},   # optional: sub-entity custom spaces
).run(parameters={"snapshot_date": "2025-12-31"})    # optional: runtime parameter overrides
```

**Config merge with custom spaces:**
- Custom fields live in named spaces (`entity.spaces["risk"]`), separate from primary fields
- Unique space field names accessible transparently (`entity.risk_score`)
- `primary_only()`, `drop_space()`, `select_space()` return new copies
- Same field name across spaces is allowed (explicit access required)

**Key files:**
- `configs/demo/facility_demo.yaml` — single-entity pipeline example
- `configs/demo/obligor_demo.yaml` — 3-level nesting example
- `configs/demo/collateral_demo.yaml` — sub-entity config
- `configs/demo/custom_risk.yaml` — custom space config (risk fields)
- `configs/demo/custom_reporting.yaml` — custom space config (reporting fields)
- `scripts/generate_sample_data.py` — data generation script